# Measuring Strategy-Decay Risk: Minimum Regime Performance and the Durability of Systematic Investing

**Authors**: N. Alexander, F. Fabozzi

**Published**: 2025-12-30

**ArXiv**: [https://arxiv.org/abs/2604.08356](https://arxiv.org/abs/2604.08356)

**Abstract**: Systematic investment strategies are exposed to a subtle but pervasive vulnerability: the progressive erosion of their effectiveness as market regimes change. Traditional risk measures, designed to capture volatility or drawdowns, overlook this form of structural fragility. This article introduces a quantitative framework for assessing the durability of systematic strategies through minimum regime performance (MRP), defined as the lowest realized risk-adjusted return across distinct historical regimes. MRP serves as a lower bound on a strategy’s robustness, capturing how performance deteriorates when underlying relationships weaken or competitive pressures compress alpha. Applied to a broad universe of established factor strategies, the measure reveals a consistent trade-off between efficiency and resilience—strategies with higher long-term Sharpe ratios do not always exhibit higher MRPs. By translating the persistence of investment efficacy into a measurable quantity, the framework provides investors with a practical diagnostic for identifying and managing strategy-decay risk, a novel dimension of portfolio fragility that complements traditional measures of market and liquidity risk.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In this phase, we define the universe of assets, set the parameters for our strategy, and state our hypothesis. This sets the stage for data collection and analysis.

In [ ]:
import numpy as np
import pandas as pd

# Configuration
universe = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'FB']  # Example universe
start_date = '2015-01-01'
end_date = '2023-01-01'
lookback_period = 252  # 1 year

# Hypothesis: The strategy's performance will degrade in certain market regimes.

## Phase 2: Data Download and Feature Computation

We download historical price data for the selected universe and compute necessary features such as returns and moving averages. Data normalization is also performed to prepare for signal generation.

In [ ]:
import yfinance as yf

# Download data
data = yf.download(universe, start=start_date, end=end_date)['Adj Close']

# Compute returns
returns = data.pct_change().dropna()

# Normalize data
normalized_data = (data - data.mean()) / data.std()

## Phase 3: Signal Generation and Portfolio Construction

Generate trading signals based on computed features and construct the portfolio. This involves determining position sizes and how the portfolio is rebalanced.

In [ ]:
# Simple moving average crossover strategy
short_window = 40
long_window = 100

signals = pd.DataFrame(index=data.index)
signals['signal'] = 0.0
signals['short_mavg'] = data.rolling(window=short_window, min_periods=1).mean()
signals['long_mavg'] = data.rolling(window=long_window, min_periods=1).mean()

# Generate signals
signals['signal'][short_window:] = np.where(signals['short_mavg'][short_window:] > signals['long_mavg'][short_window:], 1.0, 0.0)
signals['positions'] = signals['signal'].diff()

## Phase 4: Vectorized Backtest

Conduct a backtest of the strategy using vectorized operations. We shift signals by one period to avoid look-ahead bias and calculate the strategy returns.

In [ ]:
# Backtest
shifted_signals = signals['signal'].shift(1)
strategy_returns = shifted_signals * returns
cumulative_returns = (1 + strategy_returns).cumprod() - 1

## Phase 5: Performance Evaluation

Evaluate the strategy's performance using metrics such as Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. Plot the equity curve to visualize performance.

In [ ]:
import matplotlib.pyplot as plt

# Performance metrics
def calculate_sharpe_ratio(returns, risk_free_rate=0.0):
    return (returns.mean() - risk_free_rate) / returns.std()

def calculate_sortino_ratio(returns, risk_free_rate=0.0):
    downside_std = returns[returns < 0].std()
    return (returns.mean() - risk_free_rate) / downside_std

def calculate_calmar_ratio(returns):
    max_drawdown = calculate_max_drawdown(returns)
    return returns.mean() / abs(max_drawdown)

def calculate_max_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    return drawdown.min()

sharpe_ratio = calculate_sharpe_ratio(strategy_returns)
sortino_ratio = calculate_sortino_ratio(strategy_returns)
calmar_ratio = calculate_calmar_ratio(strategy_returns)
max_drawdown = calculate_max_drawdown(strategy_returns)

# Plot equity curve
plt.figure(figsize=(10, 6))
plt.plot(cumulative_returns, label='Strategy Equity Curve')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio}')
print(f'Sortino Ratio: {sortino_ratio}')
print(f'Calmar Ratio: {calmar_ratio}')
print(f'Max Drawdown: {max_drawdown}')

## Phase 6: Monitoring Stub

Set up a stub for monitoring daily P&L and positions. This function can be expanded to include more comprehensive monitoring and alerting.

In [ ]:
def monitor_daily_performance(latest_data):
    # Placeholder function for monitoring daily P&L and positions
    # latest_data: DataFrame with latest market data
    current_positions = shifted_signals.iloc[-1]  # Example of accessing the last known positions
    daily_pnl = (latest_data.pct_change() * current_positions).sum()
    print(f'Daily P&L: {daily_pnl}')
    print(f'Current Positions: {current_positions}')